In [1]:
import pandas as pd
import numpy as np

---

### **| Fase 1 ) -** Pré-Processamento

---

In [2]:
df_fair = pd.read_excel('Data/05_Sheets.xlsx')
df_fair.head()

,id,nome,url,repo,is_duplicate,key_value,parent_id
0,CRED-001,Statlog (German Credit Data),Link,UCI Repository,Yes,144,CRED-002
1,CRED-002,South German Credit,Link,Kaggle,Ofc,sid321axn/south-german-credit-updated,NaN
2,CRED-003,Default of Credit Card Clients,Link,UCI Repository,Mod,350,CRED-012
3,CRED-004,Australian Credit Approval,Link,Kaggle,No,bfueojjsjdjsl/australian-credit-approval-data-set,NaN
4,CRED-005,Japanese Credit Screening,Link,Kaggle,No,xiangshan1989/japanese-credit-screening-from-t...,NaN


In [3]:
df_cleaned = df_fair.dropna(subset=['url']).copy()
df_cleaned = df_cleaned[((df_cleaned['is_duplicate'] == 'No') | (df_cleaned['is_duplicate'] == 'Ofc'))]
df_cleaned['parent_id'] = df_cleaned['parent_id'].fillna('NaN')
df_cleaned = df_cleaned.reset_index(drop=True)

print(f"| Linhas originais: {len(df_fair)}")
print(f"| Linhas prontas para download: {len(df_cleaned)}")

| Linhas originais: 70
| Linhas prontas para download: 50


---

##### | 1.1 ) - Kaggle

In [4]:
df_cleaned_kaggle = df_cleaned[df_cleaned['repo'] == 'Kaggle']
df_cleaned_kaggle.head()

,id,nome,url,repo,is_duplicate,key_value,parent_id
0,CRED-002,South German Credit,Link,Kaggle,Ofc,sid321axn/south-german-credit-updated,NaN
1,CRED-004,Australian Credit Approval,Link,Kaggle,No,bfueojjsjdjsl/australian-credit-approval-data-set,NaN
2,CRED-005,Japanese Credit Screening,Link,Kaggle,No,xiangshan1989/japanese-credit-screening-from-t...,NaN
3,CRED-007,Polish Companies Bankruptcy,Link,Kaggle,No,stealthtechnologies/predict-bankruptcy-in-poland,NaN
9,CRED-022,Home Credit Default Risk,Link,Kaggle,No,competitions/home-credit-default-risk/data,NaN


---

##### | 1.2 ) - UCI Repository	

In [5]:
df_cleaned_UCI = df_cleaned[df_cleaned['repo'] == 'UCI Repository']
df_cleaned_UCI.head()

,id,nome,url,repo,is_duplicate,key_value,parent_id


---

##### | 1.3 ) - Open ML

In [6]:
df_cleaned_OpenML = df_cleaned[df_cleaned['repo'] == 'OpenML']
df_cleaned_OpenML.head()

,id,nome,url,repo,is_duplicate,key_value,parent_id
4,CRED-008,Qualitative Bankruptcy,Link,OpenML,No,1495,NaN
5,CRED-010,credit-approval,Link,OpenML,Ofc,29,NaN
6,CRED-012,default-of-credit-card-clients v1,Link,OpenML,Ofc,42477,NaN
7,CRED-020,LoanDefaultPrediction,Link,OpenML,No,44067,NaN
8,CRED-021,bank-marketing,Link,OpenML,No,44074,NaN


---

### **| Fase 2 ) -** Implementando API

---

In [7]:
def get_columns_size_mean(list_colums: list) -> float:
    return np.mean([len(str(col)) for col in list_colums])

def is_encrypted(x: float, threshold: int = 5) -> bool:
    return x < threshold

---

##### | 2.1 ) - Implementando para Kaggle

In [8]:
import pandas as pd
import numpy as np
import os
import shutil
import zipfile
import locale
from kaggle.api.kaggle_api_extended import KaggleApi

try:
    locale.setlocale(locale.LC_TIME, 'C')
except locale.Error:
    pass 

def process_kaggle_datasets(df_input, path='./DataBuffer'):
    api = KaggleApi()
    api.authenticate()

    if not os.path.exists(path):
        os.makedirs(path)

    results_list = []
    processed_cache = {}

    for index, row in df_input.iterrows():
        dataset_id = row['id']
        ref = str(row['key_value'])
        nome = row['nome']
        url = row['url']
        is_dup = row['is_duplicate']
        
        parent_id = str(row['parent_id']).replace('[', '').replace(']', '')

        try:
            # CORREÇÃO 2: Lógica separada para descompactar competições
            if ref.startswith('competitions/'):
                comp_name = ref.replace('competitions/', '')
                api.competition_download_files(comp_name, path=path)
                
                for item in os.listdir(path):
                    if item.endswith('.zip'):
                        zip_ref = zipfile.ZipFile(os.path.join(path, item), 'r')
                        zip_ref.extractall(path)
                        zip_ref.close()
                        os.remove(os.path.join(path, item)) # Deleta o zip limpo
            else:
                api.dataset_download_files(ref, path=path, unzip=True)
                
            files = [f for f in os.listdir(path) if f.endswith('.csv')]
            
            if not files:
                print("\n|")
                print(f"| {dataset_id} -> Nenhum arquivo CSV encontrado para '{ref}'.")
                print("|\n")
                continue
                
            file_path = os.path.join(path, files[0])
            df_temp = pd.read_csv(file_path, low_memory=False)
            
            sensitive_terms = ['age', 'gender', 'sex', 'race', 'ethnicity', 'status']
            found_sensitive = [col for col in df_temp.columns if any(s in str(col).lower() for s in sensitive_terms)]
            
            columns = df_temp.columns.values

            row_data = {
                'id': dataset_id,
                'User_Ref': ref,
                'Dataset_Title': nome,
                'URL': url,
                'is_duplicate': is_dup,
                'parent_id': parent_id if parent_id != 'nan' else 'NaN',
                'File_Name': files[0],
                'is_encrypted' : is_encrypted(x=get_columns_size_mean(columns), threshold=5),
                'Columns' : columns,
                'Columns_Count': df_temp.shape[1],
                'Rows_Count': df_temp.shape[0],
                'Rows_With_Missing': df_temp.isnull().any(axis=1).sum(),
                'Missing_Values': df_temp.isnull().sum().sum(),
                'Numeric_Cols': len(df_temp.select_dtypes(include=['number']).columns),
                'Categorical_Cols': len(df_temp.select_dtypes(exclude=['number']).columns),
                'Sensitive_Columns': found_sensitive,
                'Memory_Usage_MB': round(df_temp.memory_usage(deep=True).sum() / (1024**2), 2)
            }
            
            results_list.append(row_data)
            
            if is_dup in ['Ofc', 'No']:
                processed_cache[dataset_id] = row_data
                
        except Exception as e:
            print("\n|")
            print(f"| {dataset_id} -> Erro ao processar o link Kaggle '{ref}': {e}")
            print("|\n")
        
        finally:
            for f in os.listdir(path):
                file_to_rem = os.path.join(path, f)
                try:
                    if os.path.isfile(file_to_rem):
                        os.remove(file_to_rem)
                    elif os.path.isdir(file_to_rem):
                        shutil.rmtree(file_to_rem)
                except Exception:
                    pass

    return pd.DataFrame(results_list)

df_info_kaggle = process_kaggle_datasets(df_input=df_cleaned_kaggle)

Dataset URL: https://www.kaggle.com/datasets/sid321axn/south-german-credit-updated
Dataset URL: https://www.kaggle.com/datasets/bfueojjsjdjsl/australian-credit-approval-data-set
Dataset URL: https://www.kaggle.com/datasets/xiangshan1989/japanese-credit-screening-from-the-uc-irvine
Dataset URL: https://www.kaggle.com/datasets/stealthtechnologies/predict-bankruptcy-in-poland

|
| CRED-022 -> Erro ao processar o link Kaggle 'competitions/home-credit-default-risk/data': 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
|

Dataset URL: https://www.kaggle.com/datasets/wordsforthewise/lending-club

|
| CRED-023 -> Erro ao processar o link Kaggle 'wordsforthewise/lending-club': [Errno 21] Is a directory: './DataBuffer/accepted_2007_to_2018q4.csv'
|


|
| CRED-025 -> Erro ao processar o link Kaggle 'competitions/ieee-fraud-detection/data': 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/competitions.CompetitionApiS

In [9]:
df_info_kaggle.head()

,id,User_Ref,Dataset_Title,URL,is_duplicate,parent_id,File_Name,is_encrypted,Columns,Columns_Count,Rows_Count,Rows_With_Missing,Missing_Values,Numeric_Cols,Categorical_Cols,Sensitive_Columns,Memory_Usage_MB
0,CRED-002,sid321axn/south-german-credit-updated,South German Credit,Link,Ofc,NaN,german_credit.csv,False,"[status, duration, credit_history, purpose, am...",21,1000,0,0,3,18,"[status, personal_status_sex, age]",0.40
1,CRED-004,bfueojjsjdjsl/australian-credit-approval-data-set,Australian Credit Approval,Link,No,NaN,Credit_Card_Applications.csv,True,"[CustomerID, A1, A2, A3, A4, A5, A6, A7, A8, A...",16,690,0,0,16,0,[],0.08
2,CRED-005,xiangshan1989/japanese-credit-screening-from-t...,Japanese Credit Screening,Link,No,NaN,credit_approval_uci.csv,True,"[A1, A2, A3, A4, A5, A6, A7, A8, A9, A10, A11,...",16,690,126,435,7,9,[],0.09
3,CRED-007,stealthtechnologies/predict-bankruptcy-in-poland,Polish Companies Bankruptcy,Link,No,NaN,data.csv,True,"[A1, A2, A3, A4, A5, A6, A7, A8, A9, A10, A11,...",66,43405,23438,41322,66,0,[],21.86
4,CRED-026,mlg-ulb/creditcardfraud,Credit Card Fraud Detection (ULB),Link,Ofc,NaN,creditcard.csv,True,"[Time, V1, V2, V3, V4, V5, V6, V7, V8, V9, V10...",31,284807,0,0,31,0,[],67.36


---

##### | 2.2 ) - Implementando para UCI

In [10]:
from ucimlrepo import fetch_ucirepo

def process_uci_datasets(df_input):
    results_list = []
    processed_cache = {}

    for index, row in df_input.iterrows():
        dataset_id = row['id']
        ref = str(row['key_value'])
        nome = row['nome']
        url = row['url']
        is_dup = row['is_duplicate']
        
        parent_id = str(row['parent_id']).replace('[', '').replace(']', '')

        if is_dup == 'Yes' and parent_id in processed_cache:
            print(f"  -> Duplicata detectada. Copiando metadados da matriz {parent_id}.")
            cached_data = processed_cache[parent_id].copy()
            cached_data['id'] = dataset_id
            cached_data['Dataset_Title'] = nome
            cached_data['User_Ref'] = ref
            cached_data['URL'] = url
            cached_data['is_duplicate'] = is_dup
            cached_data['parent_id'] = parent_id
            
            results_list.append(cached_data)
            continue 

        try:
            # Baixa o dataset via API oficial da UCI usando o ID
            dataset = fetch_ucirepo(id=int(ref))
            
            # Pega o DataFrame completo (Features + Targets)
            df_temp = dataset.data.original
            
            # Proteção caso a API da UCI falhe em entregar o dataframe unificado
            if df_temp is None:
                df_temp = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
            
            sensitive_terms = ['age', 'gender', 'sex', 'race', 'ethnicity', 'status']
            found_sensitive = [col for col in df_temp.columns if any(s in str(col).lower() for s in sensitive_terms)]
            
            columns = df_temp.columns.values

            row_data = {
                'id': dataset_id,
                'User_Ref': ref,
                'Dataset_Title': nome,
                'URL': url,
                'is_duplicate': is_dup,
                'parent_id': parent_id if parent_id != 'nan' else 'NaN',
                'File_Name': 'UCI_API_Direct', # Não há arquivo físico
                'is_encrypted' : is_encrypted(x=get_columns_size_mean(columns), threshold=5),
                'Columns' : columns,
                'Columns_Count': df_temp.shape[1],
                'Rows_Count': df_temp.shape[0],
                'Rows_With_Missing': df_temp.isnull().any(axis=1).sum(),
                'Missing_Values': df_temp.isnull().sum().sum(),
                'Numeric_Cols': len(df_temp.select_dtypes(include=['number']).columns),
                'Categorical_Cols': len(df_temp.select_dtypes(exclude=['number']).columns),
                'Sensitive_Columns': found_sensitive,
                'Memory_Usage_MB': round(df_temp.memory_usage(deep=True).sum() / (1024**2), 2)
            }
            
            results_list.append(row_data)
            
            if is_dup in ['Ofc', 'No']:
                processed_cache[dataset_id] = row_data
                
        except Exception as e:
            print("\n|")
            print(f"| {dataset_id} -> Erro ao processar o link UCI '{ref}': {e}")
            print("|\n")

    return pd.DataFrame(results_list)

# Execução:
df_info_uci = process_uci_datasets(df_input=df_cleaned_UCI)

In [11]:
df_info_uci

""


---

##### | 3.3 ) - Implementando para Open ML

In [12]:
import openml

def process_openml_datasets(df_input):
    results_list = []
    processed_cache = {}

    for index, row in df_input.iterrows():
        dataset_id = row['id']
        ref = str(row['key_value'])
        nome = row['nome']
        url = row['url']
        is_dup = row['is_duplicate']
        
        parent_id = str(row['parent_id']).replace('[', '').replace(']', '')

        if is_dup == 'Yes' and parent_id in processed_cache:
            print(f"  -> Duplicata detectada. Copiando metadados da matriz {parent_id}.")
            cached_data = processed_cache[parent_id].copy()
            cached_data['id'] = dataset_id
            cached_data['Dataset_Title'] = nome
            cached_data['User_Ref'] = ref
            cached_data['URL'] = url
            cached_data['is_duplicate'] = is_dup
            cached_data['parent_id'] = parent_id
            
            results_list.append(cached_data)
            continue 

        try:
            # Baixa o dataset diretamente para a memória via API do OpenML
            dataset = openml.datasets.get_dataset(int(ref), download_data=True)
            
            # Extrai os dados já em formato Pandas DataFrame
            # X = features, y = target (juntamos os dois para análise completa)
            X, y, _, _ = dataset.get_data(dataset_format='dataframe')
            df_temp = pd.concat([X, y], axis=1)
            
            sensitive_terms = ['age', 'gender', 'sex', 'race', 'ethnicity', 'status']
            found_sensitive = [col for col in df_temp.columns if any(s in str(col).lower() for s in sensitive_terms)]
            
            columns = df_temp.columns.values

            row_data = {
                'id': dataset_id,
                'User_Ref': ref,
                'Dataset_Title': nome,
                'URL': url,
                'is_duplicate': is_dup,
                'parent_id': parent_id if parent_id != 'nan' else 'NaN',
                'File_Name': 'OpenML_API_Direct', # Não há arquivo físico
                'is_encrypted' : is_encrypted(x=get_columns_size_mean(columns), threshold=5),
                'Columns' : columns,
                'Columns_Count': df_temp.shape[1],
                'Rows_Count': df_temp.shape[0],
                'Rows_With_Missing': df_temp.isnull().any(axis=1).sum(),
                'Missing_Values': df_temp.isnull().sum().sum(),
                'Numeric_Cols': len(df_temp.select_dtypes(include=['number']).columns),
                'Categorical_Cols': len(df_temp.select_dtypes(exclude=['number']).columns),
                'Sensitive_Columns': found_sensitive,
                'Memory_Usage_MB': round(df_temp.memory_usage(deep=True).sum() / (1024**2), 2)
            }
            
            results_list.append(row_data)
            
            if is_dup in ['Ofc', 'No']:
                processed_cache[dataset_id] = row_data
                
        except Exception as e:
            print("\n|")
            print(f"|  -> Erro ao processar o link OpenML '{ref}': {e}")
            print("|\n")

    return pd.DataFrame(results_list)

# Execução:
df_info_openml = process_openml_datasets(df_input=df_cleaned_OpenML)

In [13]:
df_info_openml.head()

,id,User_Ref,Dataset_Title,URL,is_duplicate,parent_id,File_Name,is_encrypted,Columns,Columns_Count,Rows_Count,Rows_With_Missing,Missing_Values,Numeric_Cols,Categorical_Cols,Sensitive_Columns,Memory_Usage_MB
0,CRED-008,1495,Qualitative Bankruptcy,Link,No,NaN,OpenML_API_Direct,True,"[V1, V2, V3, V4, V5, V6, Class]",7,250,0,0,0,7,[],0.00
1,CRED-010,29,credit-approval,Link,Ofc,NaN,OpenML_API_Direct,True,"[A1, A2, A3, A4, A5, A6, A7, A8, A9, A10, A11,...",16,690,37,67,6,10,[],0.04
2,CRED-012,42477,default-of-credit-card-clients v1,Link,Ofc,NaN,OpenML_API_Direct,True,"[x1, x2, x3, x4, x5, x6, x7, x8, x9, x10, x11,...",24,30000,0,0,23,1,[],4.49
3,CRED-020,44067,LoanDefaultPrediction,Link,No,NaN,OpenML_API_Direct,True,"[id, f1, f2, f3, f5, f6, f7, f8, f9, f10, f13,...",736,55319,0,0,732,4,[],286.26
4,CRED-021,44074,bank-marketing,Link,No,NaN,OpenML_API_Direct,True,"[V1, V6, V10, V12, V13, V14, V15, Class]",8,10578,0,0,2,6,[],0.24


---

### **| Fase 3 ) -** Finalizando Dataset

---

---

##### | 3.1 ) - Limpeza dos dados

In [14]:
# df_final = pd.concat([df_info_kaggle, df_info_openml, df_info_uci], ignore_index=True)
df_final = pd.concat([df_info_kaggle, df_info_openml], ignore_index=True)

df_final = df_final.drop(columns=['File_Name', 'User_Ref', 'is_duplicate', 'parent_id'], errors='ignore')

df_final['Columns'] = df_final['Columns'].apply(
    lambda x: str(list(x)) if hasattr(x, '__iter__') and not isinstance(x, str) else str(x)
)

---

##### | 3.2 ) - Engenharia de atributos

In [15]:
# C.2) Taxa de Esparsidade (% de células vazias no dataset inteiro)
df_final['Sparsity_Ratio_%'] = np.where(
    (df_final['Rows_Count'] * df_final['Columns_Count']) > 0,
    round((df_final['Missing_Values'] / (df_final['Rows_Count'] * df_final['Columns_Count'])) * 100, 4),
    0
)

# D. Risco de Dimensionalidade (Features por Observação)
df_final['Dimensionality_Ratio'] = np.where(
    df_final['Rows_Count'] > 0,
    round(df_final['Columns_Count'] / df_final['Rows_Count'], 6),
    0
)

# E. Proporção Categórica (% de colunas que não são números)
df_final['Categorical_Ratio_%'] = np.where(
    df_final['Columns_Count'] > 0,
    round((df_final['Categorical_Cols'] / df_final['Columns_Count']) * 100, 2),
    0
)

# F. Indicador de Justiça Algorítmica (Contém dados demográficos?)
df_final['Has_Sensitive_Data'] = df_final['Sensitive_Columns'].apply(
    lambda x: True if isinstance(x, (list, np.ndarray)) and len(x) > 0 else False
)

# G. Contagem exata de atributos sensíveis encontrados
df_final['Sensitive_Count'] = df_final['Sensitive_Columns'].apply(
    lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0
)

---

##### | 3.3 ) - Finalização

In [16]:
df_final = df_final.sort_values(by='id').reset_index(drop=True)

caminho_saida = 'Data/06_Datasets_Metadata_Final.xlsx'
df_final.to_excel(caminho_saida, index=False, engine='openpyxl')

print(f"| Tabela final consolidada, limpa e enriquecida com {len(df_final)} datasets.")
print(f"| Arquivo salvo em: {caminho_saida}")

| Tabela final consolidada, limpa e enriquecida com 47 datasets.
| Arquivo salvo em: Data/06_Datasets_Metadata_Final.xlsx
